In [1]:
import pandas as pd

ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["userId", "movieId", "rating", "timestamp"],
    header=None
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)

In [2]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [5]:
movies["content"] = movies["title"] + " " + movies["genres"].str.replace("|", " ")
movies.head()

,movieId,title,genres,content
0,1,Toy Story (1995),Animation|Children's|Comedy,Toy Story (1995) Animation Children's Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy,Jumanji (1995) Adventure Children's Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men (1995) Comedy Romance
3,4,Waiting to Exhale (1995),Comedy|Drama,Waiting to Exhale (1995) Comedy Drama
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II (1995) Comedy


In [42]:
data = ratings.merge(movies, on="movieId")
data.head(20)

,userId,movieId,rating,timestamp,title,genres,content
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama,One Flew Over the Cuckoo's Nest (1975) Drama
1,1,661,3,978302109,James and the Giant Peach (1996),Animation|Children's|Musical,James and the Giant Peach (1996) Animation Chi...
2,1,914,3,978301968,My Fair Lady (1964),Musical|Romance,My Fair Lady (1964) Musical Romance
3,1,3408,4,978300275,Erin Brockovich (2000),Drama,Erin Brockovich (2000) Drama
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy,"Bug's Life, A (1998) Animation Children's Comedy"
5,1,1197,3,978302268,"Princess Bride, The (1987)",Action|Adventure|Comedy|Romance,"Princess Bride, The (1987) Action Adventure Co..."
6,1,1287,5,978302039,Ben-Hur (1959),Action|Adventure|Drama,Ben-Hur (1959) Action Adventure Drama
7,1,2804,5,978300719,"Christmas Story, A (1983)",Comedy|Drama,"Christmas Story, A (1983) Comedy Drama"
8,1,594,4,978302268,Snow White and the Seven Dwarfs (1937),Animation|Children's|Musical,Snow White and the Seven Dwarfs (1937) Animati...
9,1,919,4,978301368,"Wizard of Oz, The (1939)",Adventure|Children's|Drama|Musical,"Wizard of Oz, The (1939) Adventure Children's ..."


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['content'])

tfidf_matrix.shape

(3883, 4369)

In [44]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 18685 stored elements and shape (3883, 4369)>
  Coords	Values
  (0, 3996)	0.662451513042016
  (0, 3762)	0.4729987787245739
  (0, 86)	0.27799979725969426
  (0, 240)	0.37326189252954617
  (0, 810)	0.29954416585291904
  (0, 897)	0.17633806510709593
  (1, 86)	0.2926176303302647
  (1, 810)	0.31529484861191
  (1, 2153)	0.7319067807245804
  (1, 156)	0.30783801069460653
  (1, 1435)	0.4295489545554287
  (2, 86)	0.2505250503314696
  (2, 897)	0.15891055702836004
  (2, 1764)	0.6266231562056526
  (2, 2832)	0.5166665998237939
  (2, 2548)	0.4480737876198115
  (2, 3297)	0.22718617828286322
  (3, 86)	0.28042837773329216
  (3, 897)	0.17787853810704954
  (3, 4193)	0.6115171932896617
  (3, 1391)	0.7014185404308574
  (3, 1248)	0.1542008563327492
  (4, 86)	0.28049067132586025
  (4, 897)	0.17791805155882287
  (4, 1447)	0.626576037164729
  :	:
  (3876, 3926)	0.2971023304100387
  (3876, 94)	0.4056129475612573
  (3876, 729)	0.7917923497834353
  (3877

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [8]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

In [9]:
def recommend(title, k=10):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:k+1]
    
    movie_indices = [i[0] for i in sim_scores]
    return movies['title'].iloc[movie_indices]

In [10]:
recommend("Toy Story (1995)")

3045                       Toy Story 2 (1999)
3331    We're Back! A Dinosaur's Story (1993)
2892                  Story of Us, The (1999)
2039                        L.A. Story (1991)
12                               Balto (1995)
2205                    Lilian's Story (1995)
292         Pyromaniac's Love Story, A (1995)
236                     Goofy Movie, A (1995)
47                          Pocahontas (1995)
2490                   King and I, The (1999)
Name: title, dtype: object

In [11]:
user_id = 1

In [12]:
liked_movies = data[(data['userId'] == user_id) & (data['rating'] >= 4)]
liked_titles = liked_movies['title'].values
liked_titles[:5]

array(["One Flew Over the Cuckoo's Nest (1975)", 'Erin Brockovich (2000)',
       "Bug's Life, A (1998)", 'Ben-Hur (1959)',
       'Christmas Story, A (1983)'], dtype=object)

In [13]:
import numpy as np

def recommend_for_user(user_id, k=10):
    liked = data[(data['userId'] == user_id) & (data['rating'] >= 4)]
    liked_indices = [indices[title] for title in liked['title'] if title in indices]
    
    # Mean similarity across liked movies
    sim_scores = cosine_sim[liked_indices].max(axis=0)
    
    # Remove already seen movies
    seen_indices = data[data['userId'] == user_id]['movieId'].map(
        movies.set_index('movieId').index.get_loc
    )
    
    sim_scores[liked_indices] = 0
    
    top_indices = np.argsort(sim_scores)[-k:][::-1]
    
    return movies['title'].iloc[top_indices]

In [14]:
recommend_for_user(1)

274                        Miracle on 34th Street (1994)
1942                   Back to the Future Part II (1989)
1943                  Back to the Future Part III (1990)
3335                                      Titanic (1953)
2088              Chambermaid on the Titanic, The (1998)
1050              Aladdin and the King of Thieves (1996)
2559    Star Wars: Episode I - The Phantom Menace (1999)
2892                             Story of Us, The (1999)
997                                 Love Bug, The (1969)
2425                               Last Days, The (1998)
Name: title, dtype: object

In [18]:
data[data['userId'] == 1].sort_values(by='rating', ascending=False).head(10)

,userId,movieId,rating,timestamp,title,genres
0,1,1193,5,978300760,One Flew Over the Cuckoo's Nest (1975),Drama
4,1,2355,5,978824291,"Bug's Life, A (1998)",Animation|Children's|Comedy
10,1,595,5,978824268,Beauty and the Beast (1991),Animation|Children's|Musical
7,1,2804,5,978300719,"Christmas Story, A (1983)",Comedy|Drama
6,1,1287,5,978302039,Ben-Hur (1959),Action|Adventure|Drama
18,1,3105,5,978301713,Awakenings (1990),Drama
14,1,1035,5,978301753,"Sound of Music, The (1965)",Musical
40,1,1,5,978824268,Toy Story (1995),Animation|Children's|Comedy
41,1,1961,5,978301590,Rain Man (1988),Drama
36,1,1836,5,978300172,"Last Days of Disco, The (1998)",Drama


Leave-One-Out Evaluvation (Recall)

In [19]:
import numpy as np

# Get users with at least 2 liked movies
liked_data = data[data['rating'] >= 4]

user_counts = liked_data['userId'].value_counts()

valid_users = user_counts[user_counts >= 2].index

len(valid_users)

6037

In [20]:
def recommend_from_liked(liked_titles, k=10):
    liked_indices = [indices[t] for t in liked_titles if t in indices]

    if len(liked_indices) == 0:
        return []

    sim_scores = cosine_sim[liked_indices].mean(axis=0)

    sim_scores[liked_indices] = 0

    top_indices = np.argsort(sim_scores)[-k:][::-1]

    return movies['title'].iloc[top_indices].values

In [21]:
def recall_at_k(k=10, n_users=100):
    hits = 0
    evaluated = 0

    for user in valid_users[:n_users]:

        user_liked = liked_data[liked_data['userId'] == user]['title'].values

        if len(user_liked) < 2:
            continue

        # randomly hide one
        hidden = np.random.choice(user_liked)

        train_liked = [t for t in user_liked if t != hidden]

        recommendations = recommend_from_liked(train_liked, k)

        if hidden in recommendations:
            hits += 1

        evaluated += 1

    return hits / evaluated

In [41]:
recall_at_k(k=10, n_users=100)

0.0